# Reproducing Raicevic et al. (2023) from dandiset 000491

**Paper.** Raicevic N, Forer JM, Ladrón-de-Guevara A, Du T, Nedergaard M, Kelley DH, Boster K
(2023). Sizes and shapes of perivascular spaces surrounding murine pial arteries. Research Square
preprint, version 1, [doi:10.21203/rs.3.rs-2587250/v1](https://doi.org/10.21203/rs.3.rs-2587250/v1).
The claims file was written from the full text of this preprint as served by PubMed Central
(PMC9949243), with its figure legends and Tables 1 to 3; the Additional Files were not in the
text. The paper links no code.

**Dandiset.** [DANDI:000491](https://dandiarchive.org/dandiset/000491/0.230602.1307), version
0.230602.1307 (doi:10.48324/dandi.000491/0.230602.1307). The paper cites no version. This is the
most recent published version, and the only one that can support the analysis: the earlier
version 0.230427.1809 holds no PVS mask for locations L, M and N (the data check below lists the
masks each version holds). The draft has the same 14 assets and size.

**What the dandiset holds.** For each of the 14 pial artery segments (locations A to N, 9 mice),
the 3D two-photon stack and the final binary segmentations of the vessel and of the PVS. The 2D
time series used to validate the segmentation are not in it. Everything here starts from the
deposited segmentations, as the claims file says; the paper's segmentation includes manual steps
that cannot be redone.

**Method.** The pipeline P1 to P9 of the claims file's opening comment, in `pvs.py`: isotropic
sampling, a 3D skeleton of the vessel, second-order polynomial fits of the centerline, normal cross
sections, the edge exclusion, areas, the circle fit of the vessel, the ellipse, spline and
polynomial fits, and the hydraulic resistance by a finite-difference solution of the Poisson
equation (in place of MATLAB's finite elements). Every reading the claims file names is run, with
two exceptions stated in the attempt record: reading (b) of P2 (all skeleton points, 10^4 to 10^5
per segment) is run for the counts only, and the resistances of the fits are solved for the
middle erosion of P6 only. No random numbers are used.

**Run time.** About 27 minutes on four cores, almost all in the cross sections and the resistance
solves.

| Claim | Tested |
|---|---|
| cross-section-counts | tested |
| segmentation-validation-overlap | blocked: data missing (no 2D time series) |
| fig5a-pvs-area-scales-with-vessel-area | tested |
| fig5e-area-ratio-distribution | tested |
| fig5bc-area-intersegment-variation | tested |
| area-ratio-per-location | tested |
| fig5d-polynomial-closest-area-ratio | tested |
| table2-fit-area-ratio-peaks | tested |
| fig6a-resistance-power-law | tested |
| fig6b-resistance-segmentation-peak | tested |
| resistance-per-location | tested |
| fig5b-polynomial-closest-resistance | tested |
| ellipse-annulus-underestimate | tested |
| spline-resistance-five-times | tested |
| fig7-velocity-profiles | tested |
| vessel-radius-peak | tested |
| fig8-ellipse-parameters | tested |
| fig9c-pvs-two-lobes | tested |
| fig9e-pvs-width-asymmetry | tested |
| left-right-no-difference | tested |
| fig9a-heights-taper | tested |
| fig9f-pvs-concave | tested |

In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit

import pvs

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)
os.environ["OMP_NUM_THREADS"] = "1"

assets = pvs.list_assets(pvs.VERSION)
started = time.time()
TARGET = {"a": 0.648, "b": 1.0}  # claims P1 readings: isotropic voxel size in um
DATA, META = {}, {}
for p1, target in TARGET.items():
    t0 = time.time()
    DATA[p1], META[p1] = pvs.run_all(assets, target)
    print(f"P1 reading {p1} ({target} um): {len(DATA[p1])} cross sections from {DATA[p1].location.nunique()} "
          f"files in {time.time() - t0:.0f} s")
COUNTS_B = {}
for p1, target in TARGET.items():
    t0 = time.time()
    COUNTS_B[p1] = pvs.run_counts_b(assets, target)
    print(f"P2 reading b at P1 {p1}: {sum(r['n_points'] for r in COUNTS_B[p1])} skeleton points in {time.time() - t0:.0f} s")
print(f"dandiset {pvs.DANDISET} version {pvs.VERSION}, {len(assets)} assets, {time.time() - started:.0f} s in all")

## Data check

Each check of the metadata against the values is stated under its output. The stacks are static
3D images, with no time base and no trial table, so there is no clock to align; the check that
matters here is the geometry (voxel size, depth direction, which mask is which).

In [ ]:
rows = []
for a in assets:
    f = pvs.open_asset(a)
    ves, pv = pvs.load_masks(f)
    vx = pvs.voxel_size(f)
    zv = np.nonzero(ves.any((1, 2)))[0]
    zp = np.nonzero(pv.any((1, 2)))[0]
    acq = f["acquisition"]
    # vessel intensity against depth, on 8 planes through the vessel (ChanA, the vessel channel)
    d = acq["TwoPhotonSeriesChanA/data"]
    zs = np.linspace(zv[0], zv[-1], 8).astype(int)
    prof = [float(d[z, :, :, 0][ves[z]].mean()) for z in zs]
    rho = stats.spearmanr(zs, prof).statistic
    rows.append({
        "location": pvs.location_of(f), "subject": f["general/subject/subject_id"][()].decode(),
        "strain": f["general/subject/strain"][()].decode(), "sex": f["general/subject/sex"][()].decode(),
        "age": f["general/subject/age"][()].decode(), "planes": ves.shape[0], "shape_yx": ves.shape[1:],
        "voxel_um": vx, "channels": ",".join(k[-1] for k in acq), "image_dtype": str(d.dtype),
        "masks": ",".join(k[-1] for k in f["processing/ophys/ImageSegmentation"]),
        "vessel_vox": int(ves.sum()), "pvs_vox": int(pv.sum()), "overlap_vox": int((ves & pv).sum()),
        "pvs_frac_volume": round(pv.mean(), 3), "vessel_z": (int(zv[0]), int(zv[-1])),
        "pvs_z": (int(zp[0]), int(zp[-1])), "deepest_plane_pvs_frac": round(pv[zp[-1]].mean(), 2),
        "intensity_vs_depth_rho": round(rho, 2),
        "emission_nm": [float(f[f"general/optophysiology/{p}/optical_channel/emission_lambda"][()])
                        for p in f["general/optophysiology"]],
    })
info = pd.DataFrame(rows).sort_values("location").set_index("location")
print(info.to_string())
print("\nDescription: 14 datasets from 9 subjects; locations per subject:")
print(info.groupby("subject").apply(lambda d: "".join(d.index), include_groups=False).to_string())
print("\nChanA description:", f["acquisition/TwoPhotonSeriesChanA"].attrs["description"])
print("Mask descriptions:", {k: f[f"processing/ophys/ImageSegmentation/{k}/image_mask"].attrs["description"]
                             for k in f["processing/ophys/ImageSegmentation"]})
for a in assets:
    f = pvs.open_asset(a)
    if pvs.voxel_size(f)[2] != 1.0:
        print(f"Session description ({pvs.location_of(f)}, stated voxel depth {pvs.voxel_size(f)[2]} um):",
              f["session_description"][()].decode())

Checks of the metadata against the values:

- Nine subjects and 14 files, one per location, as the description maps them. Every file has a
  vessel mask (ChanA) and a PVS mask (ChanC). Image channels are float64 in "arbitrary unit".
- Depth direction. The mean vessel-channel intensity inside the vessel mask falls with plane index
  in most files (negative rank correlation above), as the Methods expect from attenuation with
  depth, so plane 0 is the surface side. This is consistent with, not proof of, the orientation.
- The PVS mask fills most of the deepest planes in some files (`deepest_plane_pvs_frac`); see the
  cross-section checks below.
- Overlap: the Methods say overlaps between the vessel and PVS segmentations were removed by hand;
  the deposited masks still overlap in most files (`overlap_vox`).
- Emission wavelengths: the file gives 698 nm for ChanA, described as FITC-dextran, and 525 nm for
  ChanC, the PVS tracer; the Methods say the green (525 nm) channel holds the FITC-dextran.

In [ ]:
# Voxel size in depth. L, M and N state 0.648 um per plane, the others 1 um, while every session
# description says the planes are 1 um apart. A pial artery is close to round in cross section,
# so the height-to-width ratio of the vessel in the normal cross sections tests the stated depth.
d = DATA["a"]
d = d[d.keep_b & (d.ves_area > 0)]
rnd = d.assign(hw=d.ves_h / d.ves_w).groupby("location").hw.median()
print("Median vessel height / width in normal cross sections, with each file's stated voxel depth:")
print(pd.DataFrame({"height_over_width": rnd.round(2), "stated_dz_um": info.voxel_um.map(lambda v: v[2])}).to_string())
print("\nDigests of the masks (duplicate recordings would share one):")
dig = META["a"].digest
print("distinct:", dig.nunique(), "of", len(dig))
# Versions: what each published version holds
for v in ("0.230427.1809", pvs.VERSION):
    held = {}
    for a in pvs.list_assets(v):
        f = pvs.open_asset(a)
        held[pvs.location_of(f)] = ",".join(k[-1] for k in f["processing/ophys/ImageSegmentation"])
    print(f"version {v}: masks per location", dict(sorted(held.items())))

If the depth were mis-stated by the ratio 1 / 0.648, the vessel in those files would look 1.54
times taller or shorter than wide. The ratios above say whether the stated voxel sizes are
consistent with round vessels. The earlier published version lacks the PVS masks of L, M and N,
so it cannot support the analysis; the pinned version holds all 14.

In [ ]:
# Cross sections per location and what the exclusion of P4 keeps
P4 = ("a", "b")
P6 = pvs.EROSIONS
P7 = ("a", "b")
summ = []
for p1 in TARGET:
    g = DATA[p1].groupby("location")
    summ.append(pd.DataFrame({"P1": p1, "points": g.size(), "kept_P4a": g.keep_a.sum(), "kept_P4b": g.keep_b.sum(),
                              "med_width_a_um": META[p1].med_a.round(1), "med_width_b_um": META[p1].med_b.round(1),
                              "no_vessel_fit": g.R1_e3.apply(lambda s: int(s.isna().sum())),
                              "no_poly_fit": g.HC_e3.apply(lambda s: int(s.isna().sum()))}))
print(pd.concat(summ).to_string())


def sel(p1, p4):
    d = DATA[p1]
    return d[d["keep_" + p4]]


def ratio_seg(d, p5, k=3):
    """Segmentation area ratio: P5 reading a, segmented vessel area; b, the fitted circle."""
    return d.pvs_area / (d.ves_area if p5 == "a" else np.pi * d[f"R1_e{k}"] ** 2)


def within(x, value, tol):
    return bool(np.isfinite(x) and abs(x - value) <= tol + 1e-12)


d = sel("a", "b")
share = d.overlap_area / (d.pvs_area + d.overlap_area)
print("Pixels in both masks, as a share of the PVS mask in each slice (P1 a, P4 b): median",
      round(share.median(), 4), "; slices above 5%:", int((share > 0.05).sum()), "of", len(d), "; by location above 5%:",
      share.gt(0.05).groupby(d.location).sum().to_dict())
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(d.pvs_area, bins=60); ax[0].set_xlabel("PVS area (um^2)")
ax[1].hist(d.ves_area, bins=60); ax[1].set_xlabel("vessel area (um^2)")
ax[2].hist(d.R1_e3.dropna(), bins=60); ax[2].set_xlabel("R1 (um), erosion 3")
plt.tight_layout(); plt.show()
print("Ranges (P1 a, P4 b): PVS area", d.pvs_area.min().round(1), "to", d.pvs_area.max().round(1),
      "; vessel area", d.ves_area.min().round(1), "to", d.ves_area.max().round(1),
      "; R1", d.R1_e3.min().round(1), "to", d.R1_e3.max().round(1), "um")

## cross-section-counts

> After excluding cross sections near the edge of the imaging volume, each of the 14 PVS segments
> yields between 54 and 451 normal cross sections, about 0.8 um apart.

Every combination of P1 (voxel 0.648 or 1 um), P2 (longest path or all skeleton points) and P4
(median width from the horizontal extent of the PVS, or from the polynomial fit). Holds when the
fewest and most per segment lie in 48.6 to 496.1 and the mean spacing is 0.72 to 0.88 um.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        g = DATA[p1].groupby("location")["keep_" + p4].sum()
        sp = META[p1].spacing.mean()
        rows.append({"P1": p1, "P2": "a", "P4": p4, "fewest": int(g.min()), "most": int(g.max()),
                     "segments_with_0": int((g == 0).sum()), "mean_spacing_um": round(sp, 3)})
    for p4 in P4:
        n = []
        for r in COUNTS_B[p1]:
            med = r["med_hext"] if p4 == "a" else r["med_wpol"]
            n.append(int((r["edge_dist"] >= med / 2.5).sum()))
        sp = np.mean([r["spacing"] for r in COUNTS_B[p1]])
        rows.append({"P1": p1, "P2": "b", "P4": p4, "fewest": min(n), "most": max(n),
                     "segments_with_0": sum(x == 0 for x in n), "mean_spacing_um": round(sp, 3)})
cc = pd.DataFrame(rows)
cc["counts_in_range"] = (cc.fewest >= 54 - 5.4) & (cc.most <= 451 + 45.1)
cc["spacing_ok"] = (cc.mean_spacing_um - 0.8).abs() <= 0.08
cc["holds"] = cc.counts_in_range & cc.spacing_ok
print(cc.to_string(index=False))
print("\nPer location, P2 a (path order spacing in um):")
print(pd.concat({p1: META[p1][["points", "spacing"]].round(3) for p1 in TARGET}, axis=1).to_string())
print("\nP2 b points per location:", {p1: {r["location"]: r["n_points"] for r in COUNTS_B[p1]} for p1 in TARGET})

**Result: not reproduced.** No combination of P1, P2 and P4 holds. With the longest path (P2 a)
and the polynomial width (P4 b), the segments give 390 to 669 cross sections at 0.648 um, with a
mean spacing of 0.875 um (within tolerance), and 248 to 415 at 1 um, inside the range widened by
10%, but with a mean spacing of 1.347 um. With the horizontal extent of the PVS (P4 a), 5 segments
keep no cross section. All skeleton points (P2 b) give up to 76393 points in a segment.

## fig5a-pvs-area-scales-with-vessel-area

> PVS area increases with vessel area only approximately; the least-squares line of PVS area on
> vessel area over all cross sections has slope 1.05 (95% CI 0.973 to 1.12) and R2 of 0.158.

Ordinary least squares on the segmented areas of every retained cross section, (a) with an
intercept, (b) through the origin. Slope within 0.077 of 1.05 and R2 within 0.0158 of 0.158. R2
through the origin is computed against the mean (centred), so that it is comparable with (a); the
uncentred value is also shown.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        x, y = d.ves_area.values, d.pvs_area.values
        r = stats.linregress(x, y)
        tq = stats.t.ppf(0.975, len(x) - 2)
        b0 = (x @ y) / (x @ x)
        res = y - b0 * x
        se0 = np.sqrt(res @ res / (len(x) - 1) / (x @ x))
        r2c = 1 - (res @ res) / ((y - y.mean()) @ (y - y.mean()))
        r2u = 1 - (res @ res) / (y @ y)
        for rd, slope, lo, hi, r2 in (("a", r.slope, r.slope - tq * r.stderr, r.slope + tq * r.stderr, r.rvalue ** 2),
                                      ("b", b0, b0 - tq * se0, b0 + tq * se0, r2c)):
            rows.append({"P1": p1, "P4": p4, "reading": rd, "n": len(x), "slope": round(slope, 3),
                         "ci_low": round(lo, 3), "ci_high": round(hi, 3), "R2": round(r2, 3),
                         "R2_uncentred": round(r2u, 3) if rd == "b" else np.nan,
                         "intercept_um2": round(r.intercept, 1) if rd == "a" else 0.0,
                         "match": within(slope, 1.05, 0.077) and within(r2, 0.158, 0.0158)})
f5a = pd.DataFrame(rows)
print(f5a.to_string(index=False))
d = sel("a", "b")
print("\nPer location (P1 a, P4 b), slope with intercept:")
print(d.groupby("location").apply(lambda g: pd.Series({"n": len(g), "slope": stats.linregress(g.ves_area, g.pvs_area).slope,
      "R2": stats.linregress(g.ves_area, g.pvs_area).rvalue ** 2}), include_groups=False).round(3).to_string())
fig, ax = plt.subplots(figsize=(5, 4))
for loc, g in d.groupby("location"):
    ax.scatter(g.ves_area, g.pvs_area, s=2, label=loc)
ax.set_xlabel("vessel area (um^2)"); ax.set_ylabel("PVS area (um^2)"); ax.legend(markerscale=4, fontsize=6, ncol=2)
plt.show()
print("Plotted ranges: vessel", d.ves_area.min().round(), "to", d.ves_area.max().round(), "; PVS", d.pvs_area.min().round(), "to", d.pvs_area.max().round(), "um^2; no points outside the axes")

**Result: not reproduced.** Over the eight combinations the slope is 1.504 to 2.718, against
1.05, and R2 is 0.130 to 0.559, against 0.158. With P4 (b), R2 is 0.130 to 0.149, near the
paper's, but the slope is 2.220 to 2.718. The slopes within single segments run from -2.167 to
2.165 (P1 a, P4 b).

## fig5e-area-ratio-distribution

> The PVS-to-vessel area ratio of the segmentation varies widely across cross sections; its
> distribution peaks at 1.12, its median is 1.88 and its IQR is 1.79 (25th percentile 1.21, 75th
> 3.00), so the ratio varies by more than 100%.

Every combination of P1, P4, P5 (segmented vessel area, or the fitted circle for each erosion of
P6) and P7 (bins over all values, or over the values without outliers).

In [ ]:
EXP_5E = [("peak", 1.12, 0.112), ("median", 1.88, 0.188), ("IQR", 1.79, 0.179), ("q25", 1.21, 0.121), ("q75", 3.0, 0.3)]
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for p5, k in [("a", 3)] + [("b", k) for k in P6]:
            r = ratio_seg(d, p5, k).replace([np.inf], np.nan).dropna()
            q25, q50, q75 = np.percentile(r, [25, 50, 75])
            for p7 in P7:
                v = {"peak": pvs.peak(r, p7), "median": q50, "IQR": q75 - q25, "q25": q25, "q75": q75}
                ok = {n: within(v[n], e, t) for n, e, t in EXP_5E}
                rows.append({"P1": p1, "P4": p4, "P5": p5, "P6": k if p5 == "b" else "-", "P7": p7, "n": len(r),
                             **{n: round(v[n], 3) for n in v}, "all_within": all(ok.values()),
                             "median_IQR_within": ok["median"] and ok["IQR"]})
f5e = pd.DataFrame(rows)
print(f5e.to_string(index=False))
print("\nRatio range (P1 a, P4 b, P5 a):", ratio_seg(sel("a", "b"), "a").min().round(3), "to", ratio_seg(sel("a", "b"), "a").max().round(2))
print("Per location median ratio (P1 a, P4 b, P5 a):", ratio_seg(sel("a", "b"), "a").groupby(sel("a", "b").location).median().round(2).to_dict())

**Result: partially reproduced.** No combination puts all five values within tolerance. Under
P1 (a), P4 (b) and P5 (a) the median is 1.718 and the IQR 1.831, both within tolerance, and the
75th percentile is 2.878, also within; the 25th percentile is 1.046 (expected 1.21 within 0.121)
and the peak is 1.546 with outliers left out (P7 b) or 254.6 over all values (P7 a), against
1.12. The ratio does vary by more than 100% in the paper's sense (IQR 1.831 on a median of
1.718). With P4 (a) the median is 1.307 to 1.467, outside tolerance.

## fig5bc-area-intersegment-variation

> Variation in PVS area is much larger between PVS segments than along the length of a single
> segment.

Per segment, the median and IQR of segmented PVS area. Reading (a): the IQR of the 14 segment
medians is larger than the median of the 14 within-segment IQRs; (b): the range of the 14 medians
is larger than the largest within-segment IQR.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        g = d.groupby("location").pvs_area
        med = g.median(); iqr = g.quantile(0.75) - g.quantile(0.25)
        med, iqr = med.dropna(), iqr.dropna()
        a = (med.quantile(0.75) - med.quantile(0.25)) > iqr.median()
        b = (med.max() - med.min()) > iqr.max()
        rows.append({"P1": p1, "P4": p4, "segments": len(med), "IQR_of_medians": round(med.quantile(0.75) - med.quantile(0.25), 1),
                     "median_within_IQR": round(iqr.median(), 1), "range_of_medians": round(med.max() - med.min(), 1),
                     "largest_within_IQR": round(iqr.max(), 1), "reading_a": a, "reading_b": b})
        if p1 == "a":
            print(f"P1 {p1}, P4 {p4}: per segment median and IQR of PVS area (um^2)")
            print(pd.DataFrame({"n": g.size(), "median": med.round(1), "IQR": iqr.round(1)}).T.to_string())
f5bc = pd.DataFrame(rows)
print(f5bc.to_string(index=False))

**Result: reproduced, resting on an assumption.** Both readings hold in three of the four
combinations of P1 and P4; with P1 (a) and P4 (b), the IQR of the 14 segment medians is 4747.9
um^2 against a median within-segment IQR of 1519.8, and the range of the medians is 30496.8 against
the largest within-segment IQR of 13614.7 (E). With P1 (a) and P4 (a) only reading (a) holds (range
of medians 7253.4 against 7968.5 at H), and that combination keeps only 9 segments. P4 (b) is
adopted, as P4 (a) leaves 5 of the 14 segments without a cross section and the paper analysed all
14.

## area-ratio-per-location

> The peak of the segmented area ratio distribution ranges from 0.6 to 11.43 across the 14
> locations, and in all but one location the ratio's variation along the vessel lies within 30% of
> the peak, so variation between segments dominates.

Per location, P5 and P7 as above. "Within 30% of the peak": (a) the 25th and 75th percentiles both
lie within 30% of the peak; (b) IQR at most 30% of the peak; (c) IQR at most 60% of the peak; (d)
all non-outlier values within 30% of the peak. Expected: smallest peak 0.54 to 0.66, largest 10.29
to 12.57, count of locations 11.7 to 14.3.

In [ ]:
def per_location_peaks(d, values, p7, frac, readings):
    out = []
    for loc, v in values.groupby(d.location):
        v = v.replace([np.inf], np.nan).dropna()
        if len(v) < 5:
            continue
        pk = pvs.peak(v, p7)
        q25, q75 = np.percentile(v, [25, 75])
        nof = pvs.outlier_free(v)
        r = {"location": loc, "n": len(v), "peak": pk, "q25": q25, "q75": q75,
             "a": (abs(q25 - pk) <= frac * pk) and (abs(q75 - pk) <= frac * pk),
             "b": (q75 - q25) <= frac * pk, "c": (q75 - q25) <= 2 * frac * pk,
             "d": bool(np.all(np.abs(nof - pk) <= frac * pk))}
        out.append({k: r[k] for k in ["location", "n", "peak", "q25", "q75"] + list(readings)})
    return pd.DataFrame(out)


rows, shown = [], None
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for p5, k in [("a", 3)] + [("b", k) for k in P6]:
            for p7 in P7:
                t = per_location_peaks(d, ratio_seg(d, p5, k), p7, 0.30, "abcd")
                if (p1, p4, p5, p7) == ("a", "b", "a", "b"):
                    shown = t
                for rd in "abcd":
                    cnt = int(t[rd].sum())
                    rows.append({"P1": p1, "P4": p4, "P5": p5, "P6": k if p5 == "b" else "-", "P7": p7, "reading": rd,
                                 "locations": len(t), "min_peak": round(t.peak.min(), 3), "max_peak": round(t.peak.max(), 2),
                                 "count_within": cnt,
                                 "match": within(t.peak.min(), 0.6, 0.06) and within(t.peak.max(), 11.43, 1.143) and within(cnt, 13, 1.3)})
arl = pd.DataFrame(rows)
print("Per location, P1 a, P4 b, P5 a, P7 b:")
print(shown.round(3).to_string(index=False))
print(arl.to_string(index=False))
print("\nCombinations matching:", int(arl.match.sum()), "of", len(arl))

**Result: not reproduced.** None of the 128 combinations matches. The smallest per-location peak
is between 0.052 and 0.794 and the largest between 1.8 and 257.17, depending on the combination;
under P1 (a), P4 (b), P5 (a) and P7 (b) they are 0.052 (C) and 9.81 (G). At most 11 locations
meet a reading of "within 30% of the peak" (reading c, the loosest), and under reading (d) none
does in any combination with all 14 segments.

## fig5d-polynomial-closest-area-ratio

> Of the three fits, the polynomial fit matches the segmentation's area ratio most closely in 73%
> of cross sections, the spline fit in 19% and the ellipse fit in 8%, and its distribution matches
> the segmentation's best.

Per cross section, the fit whose area ratio is closest to the segmentation's. Every combination of
P5 (for the segmentation and, separately, for the fits), P6 and P9 (ellipse region a, ellipse
minus its overlap with the circle; b, ellipse area minus circle area), under each P1 and P4. Only
cross sections where all three fits exist are counted. Shares within 7.3, 1.9 and 0.8 points of
73, 19 and 8%.

In [ ]:
def fit_ratios(d, k, p5_fit, p9):
    den = np.pi * d[f"R1_e{k}"] ** 2 if p5_fit == "a" else d.ves_area
    ell = d[f"ell_area_a_e{k}"] if p9 == "a" else d[f"ell_area_b_e{k}"]
    return pd.DataFrame({"polynomial": d[f"pol_area_e{k}"] / den, "spline": d[f"spl_area_e{k}"] / den,
                         "ellipse": ell / den})


rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            for p5s in "ab":
                seg = ratio_seg(d, p5s, k)
                for p5f in "ab":
                    for p9 in "ab":
                        fr = fit_ratios(d, k, p5f, p9)
                        ok = fr.notna().all(axis=1) & np.isfinite(seg)
                        best = (fr[ok].sub(seg[ok], axis=0)).abs().idxmin(axis=1)
                        sh = best.value_counts(normalize=True).reindex(["polynomial", "spline", "ellipse"]).fillna(0) * 100
                        rows.append({"P1": p1, "P4": p4, "P6": k, "P5_seg": p5s, "P5_fit": p5f, "P9": p9, "n": int(ok.sum()),
                                     "dropped": int((~ok).sum()), **sh.round(1).to_dict(),
                                     "match": within(sh.polynomial, 73, 7.3) and within(sh.spline, 19, 1.9) and within(sh.ellipse, 8, 0.8),
                                     "partial": sh.polynomial == sh.max() and sh.polynomial > 50})
f5d = pd.DataFrame(rows)
print(f5d.to_string(index=False))
print("\nCombinations matching:", int(f5d.match.sum()), "; meeting the partial rule:", int(f5d.partial.sum()), "of", len(f5d))

**Result: partially reproduced.** The polynomial fit is the closest fit in 40.2 to 74.8% of
cross sections, the spline in 19.0 to 47.5% and the ellipse in 0.0 to 28.7%; no combination puts
all three shares within tolerance. The polynomial share is the largest and above 50% in 92 of the
96 combinations (with P1 a, P4 b and erosion 3: 53.7 to 65.8%, spline 20.7 to 36.5%, ellipse 1.9
to 21.5%).

## table2-fit-area-ratio-peaks

> The area ratio distribution peaks at 1.21 for the polynomial fit (close to the segmentation's
> 1.12), 2.08 for the ellipse fit, which does not narrow at the ends, and 1.48 for the spline fit,
> whose ratio varies little because it scales with vessel size alone.

Peaks, medians and the spline IQR of the fits' area ratios under each reading of P5 for the fits,
P6, P7, P9 (ellipse region), P1 and P4.

In [ ]:
EXP_T2 = [("pk_pol", 1.21, 0.121), ("pk_ell", 2.08, 0.208), ("pk_spl", 1.48, 0.148), ("md_pol", 2.15, 0.215),
          ("md_ell", 2.89, 0.289), ("md_spl", 1.58, 0.158), ("iqr_spl", 0.3, 0.03)]
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            for p5f in "ab":
                for p9 in "ab":
                    fr = fit_ratios(d, k, p5f, p9).replace([np.inf, -np.inf], np.nan)
                    iqr = {c: np.nanpercentile(fr[c], 75) - np.nanpercentile(fr[c], 25) for c in fr}
                    for p7 in P7:
                        v = {"pk_pol": pvs.peak(fr.polynomial.dropna(), p7), "pk_ell": pvs.peak(fr.ellipse.dropna(), p7),
                             "pk_spl": pvs.peak(fr.spline.dropna(), p7), "md_pol": fr.polynomial.median(),
                             "md_ell": fr.ellipse.median(), "md_spl": fr.spline.median(), "iqr_spl": iqr["spline"]}
                        rows.append({"P1": p1, "P4": p4, "P6": k, "P5_fit": p5f, "P9": p9, "P7": p7,
                                     **{n: round(x, 3) for n, x in v.items()},
                                     "iqr_pol": round(iqr["polynomial"], 3), "iqr_ell": round(iqr["ellipse"], 3),
                                     "n_within": sum(within(v[n], e, t) for n, e, t in EXP_T2),
                                     "match": all(within(v[n], e, t) for n, e, t in EXP_T2),
                                     "partial": (v["pk_ell"] > v["pk_spl"] > v["pk_pol"]) and iqr["spline"] < min(iqr["polynomial"], iqr["ellipse"])})
t2 = pd.DataFrame(rows)
print(t2.to_string(index=False))
print("\nCombinations meeting the partial rule:")
print(t2[t2.partial].to_string(index=False))
print("\nCombinations matching:", int(t2.match.sum()), "; meeting the partial rule:", int(t2.partial.sum()), "of", len(t2),
      "; most values within tolerance in one combination:", int(t2.n_within.max()), "of 7")

**Result: partially reproduced.** No combination puts all seven values within tolerance; the most
in one combination is 6 of 7. The spline fit's ratio is nearly constant (peak 1.48 to 1.64,
median 1.505 to 1.705, IQR 0.019 to 0.298), as the paper says. The order of the peaks, ellipse
above spline above polynomial, with the spline IQR the smallest, holds in 9 of 96 combinations,
eight of them with P4 (a), where the polynomial peak falls at 0.91, below the expected 1.21.

## fig6a-resistance-power-law

> The hydraulic resistance per unit length of the segmented PVS is well predicted by a power law in
> PVS area (R2 = 0.944), with exponent -1.75 (95% CI -1.763 to -1.737), close to the -2 of a shape
> scaled in size alone.

Readings: (a) least squares of log resistance on log area, R2 on the log scale; (b) nonlinear least
squares of resistance on area, R2 on the linear scale. Exponent within 0.013 of -1.75, R2 within
0.0944 of 0.944. The solver check of P8 (a circle of the same area within 1% of the exact
resistance) is shown first.

In [ ]:
# Solver check (Methods, Hydraulic resistance): circles of the areas met here
chk = []
for p1, px in TARGET.items():
    for area_um2 in (300, 1000, 3000, 10000):
        rpx = np.sqrt(area_um2 / np.pi) / px
        n = int(2 * rpx + 20)
        m = pvs.disk((n, n), n / 2, n / 2, rpx)
        R, _, _ = pvs.poiseuille(m, px)
        a = (np.sqrt(m.sum() / np.pi) - 1.5) * px * 1e-6  # the domain: 1.5 pixels inside the mask
        chk.append({"P1": p1, "area_um2": area_um2, "error_vs_exact": round(R / (8 * pvs.MU / (np.pi * a ** 4)) - 1, 4)})
print(pd.DataFrame(chk).to_string(index=False))
d = sel("a", "b")
eq_r = np.sqrt(d.pvs_area / np.pi)
print("Median PVS area", round(d.pvs_area.median()), "um^2; median radius of the circle of the same area", round(eq_r.median(), 1), "um")

rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        ok = np.isfinite(d.R_seg) & (d.R_seg > 0) & (d.pvs_area > 0)
        A, R = d.pvs_area[ok].values * 1e-12, d.R_seg[ok].values
        r = stats.linregress(np.log(A), np.log(R))
        tq = stats.t.ppf(0.975, len(A) - 2)
        rows.append({"P1": p1, "P4": p4, "reading": "a", "n": len(A), "exponent": round(r.slope, 4),
                     "ci_low": round(r.slope - tq * r.stderr, 4), "ci_high": round(r.slope + tq * r.stderr, 4),
                     "R2": round(r.rvalue ** 2, 4), "match": within(r.slope, -1.75, 0.013) and within(r.rvalue ** 2, 0.944, 0.0944),
                     "partial": within(r.rvalue ** 2, 0.944, 0.0944) and -2 <= r.slope <= -1.5})
        pp, cov = curve_fit(lambda a, c, k: c * a ** k, A / A.mean(), R / R.mean(), p0=(1, r.slope), maxfev=20000)
        pred = pp[0] * (A / A.mean()) ** pp[1] * R.mean()
        r2 = 1 - np.sum((R - pred) ** 2) / np.sum((R - R.mean()) ** 2)
        se = np.sqrt(cov[1, 1])
        rows.append({"P1": p1, "P4": p4, "reading": "b", "n": len(A), "exponent": round(pp[1], 4),
                     "ci_low": round(pp[1] - 1.96 * se, 4), "ci_high": round(pp[1] + 1.96 * se, 4), "R2": round(r2, 4),
                     "match": within(pp[1], -1.75, 0.013) and within(r2, 0.944, 0.0944),
                     "partial": within(r2, 0.944, 0.0944) and -2 <= pp[1] <= -1.5})
f6a = pd.DataFrame(rows)
print(f6a.to_string(index=False))
d = sel("a", "b")
print("\nPer location log-log exponent (P1 a, P4 b):")
print(d.groupby("location").apply(lambda g: stats.linregress(np.log(g.pvs_area), np.log(g.R_seg)).slope, include_groups=False).round(3).to_dict())
fig, ax = plt.subplots(figsize=(5, 4))
fin = d[np.isfinite(d.R_seg) & (d.pvs_area > 0)]
ax.loglog(fin.pvs_area, fin.R_seg, ".", ms=2); ax.set_xlabel("PVS area (um^2)"); ax.set_ylabel("R (Pa s/m^4)"); plt.show()
print("Plotted ranges: area", fin.pvs_area.min().round(1), "to", fin.pvs_area.max().round(), "um^2; R", f"{fin.R_seg.min():.3g} to {fin.R_seg.max():.3g};",
      len(d) - len(fin), "cross sections with no flow domain (infinite resistance) not plotted or fitted")

**Result: partially reproduced.** No combination gives the exponent within 0.013 of -1.75.
The log-log fit (reading a) gives exponents of -1.965 to -2.258 with R2 0.941 to 0.956; with P1 (a)
and P4 (a) the exponent is -1.965 (95% CI -1.981 to -1.949) and R2 0.941, which meets the partial
rule (R2 within tolerance, exponent between -2 and -1.5). With P4 (b) the exponent is -2.134
(P1 a) and -2.258 (P1 b). The nonlinear fit (reading b) gives -1.06 to -6.44. The solver is within
2.5% of the exact resistance for circles of 300 to 10000 um^2 (within 1% from 1000 um^2 at 0.648
um). Per-location exponents run from -2.806 to -1.315 (not computed for C and I, which hold cross
sections with no flow domain).

## fig6b-resistance-segmentation-peak

> The distribution of segmented PVS hydraulic resistance per unit length over all cross sections
> peaks at 1.73e15 Pa s/m^4, with a median of 3.84e15 Pa s/m^4.

P7 readings, with the 20 bins even on a linear (a) or logarithmic (b) scale. Peak within 1.73e14
of 1.73e15, median within 3.84e14 of 3.84e15.

In [ ]:
def rpeak(x, p7, scale):
    return pvs.peak(np.asarray(x, float), p7, log=(scale == "log"))


rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        R = d.R_seg[np.isfinite(d.R_seg)]
        for p7 in P7:
            for sc in ("linear", "log"):
                pk, md_ = rpeak(R, p7, sc), R.median()
                rows.append({"P1": p1, "P4": p4, "P7": p7, "bins": sc, "n": len(R), "peak": f"{pk:.3g}", "median": f"{md_:.3g}",
                             "match": within(pk, 1.73e15, 1.73e14) and within(md_, 3.84e15, 3.84e14),
                             "median_within": within(md_, 3.84e15, 3.84e14)})
f6b = pd.DataFrame(rows)
print(f6b.to_string(index=False))
d = sel("a", "b")
print("\nPer location median R_seg (P1 a, P4 b):", {k: f"{v:.3g}" for k, v in d.groupby("location").R_seg.median().items()})

**Result: partially reproduced.** The median is within tolerance only with P1 (a) and P4 (b),
4.16e15 against 3.84e15 Pa s/m^4. The peak is not within tolerance in any combination; with P1 (a)
and P4 (b) it is 1.25e15 (linear bins, outliers out), 1.94e15 or 2.01e15 (logarithmic bins),
against 1.73e15 within 1.73e14.

## resistance-per-location

> Variation in resistance is much larger between locations than along each PVS; the per-location
> peaks range from 3.23e14 to 1.98e16, and in 65% of vessels all the variation lies within 37% of
> the peak.

Per location, P7 readings on linear bins. "All the variation within 37% of the peak": (a) 25th
and 75th percentiles within 37% of the peak; (b) all non-outlier values within 37%; (c) IQR at
most 37% of the peak. The readings are labelled as in the claims file.

In [ ]:
rows, shown = [], None
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for p7 in P7:
            t = per_location_peaks(d, d.R_seg, p7, 0.37, "abd")
            t = t.rename(columns={"b": "c_iqr", "d": "b_all"}).rename(columns={"c_iqr": "c", "b_all": "b"})
            if (p1, p4, p7) == ("a", "b", "b"):
                shown = t
            for rd in "abc":
                share = 100 * t[rd].mean()
                rows.append({"P1": p1, "P4": p4, "P7": p7, "reading": rd, "locations": len(t),
                             "min_peak": f"{t.peak.min():.3g}", "max_peak": f"{t.peak.max():.3g}", "share_pct": round(share, 1),
                             "match": within(t.peak.min(), 3.23e14, 3.23e13) and within(t.peak.max(), 1.98e16, 1.98e15) and within(share, 65, 6.5)})
rpl = pd.DataFrame(rows)
print("Per location (P1 a, P4 b, P7 b):")
print(shown.assign(peak=shown.peak.map("{:.3g}".format), q25=shown.q25.map("{:.3g}".format), q75=shown.q75.map("{:.3g}".format)).to_string(index=False))
print(rpl.to_string(index=False))

**Result: not reproduced.** No combination matches. With P4 (b) the smallest per-location peak
is 3.7e13 to 5.4e13 (E), against 3.23e14, and the largest 7e16 to 2e20, against 1.98e16; the
share of locations within 37% of their peak is 0 to 50%. The per-location medians differ by more
than three orders of magnitude (5.41e13 at E to 1.16e17 at C), so between-location variation is
large, as the paper says.

## fig5b-polynomial-closest-resistance

> The distribution of hydraulic resistance of the polynomial fit matches that of the segmentation
> most closely, followed by the ellipse fit, and the polynomial fit is the best match for the
> majority of cross sections; the distribution peaks are 1.44e15 (polynomial), 7.06e14 (ellipse),
> 1.07e16 (spline) and 1.32e15 (optimal annulus) Pa s/m^4.

The fits' resistances use the circle fit at erosion 3 (see the deviations). The annulus follows
Equation 3 as the claims file reads it, with K the segmentation's area ratio (P5 a or b) and R1 at
erosion 3. Competing models: (a) the three fits, (b) the fits and the annulus. Peaks on linear
bins (P7); the logarithmic bins are shown as a variant.

In [ ]:
def models(d, p5):
    K = ratio_seg(d, p5, 3)
    return pd.DataFrame({"segmentation": d.R_seg, "polynomial": d.R_pol_e3, "ellipse": d.R_ell_e3, "spline": d.R_spl_e3,
                         "annulus": pvs.optimal_annulus(K, d.R1_e3 * 1e-6)}).replace([np.inf, -np.inf], np.nan)


EXP_PK = {"polynomial": (1.44e15, 1.44e14), "ellipse": (7.06e14, 7.06e13), "spline": (1.07e16, 1.07e15), "annulus": (1.32e15, 1.32e14)}
rows, rows_v = [], []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for p5 in "ab":
            M = models(d, p5)
            ok = M.notna().all(axis=1)
            M = M[ok]
            for comp, names in (("a", ["polynomial", "ellipse", "spline"]), ("b", ["polynomial", "ellipse", "spline", "annulus"])):
                best = M[names].sub(M.segmentation, axis=0).abs().idxmin(axis=1).value_counts(normalize=True).reindex(names).fillna(0) * 100
                for p7 in P7:
                    for sc in ("linear", "log"):
                        pk = {n: rpeak(M[n], p7, sc) for n in ["segmentation"] + list(EXP_PK)}
                        r = {"P1": p1, "P4": p4, "P5": p5, "models": comp, "P7": p7, "bins": sc, "n": len(M), "dropped": int((~ok).sum()),
                             **{f"best_{n[:4]}": round(best.get(n, np.nan), 1) for n in ["polynomial", "ellipse", "spline", "annulus"]},
                             **{f"pk_{n[:4]}": f"{pk[n]:.3g}" for n in pk},
                             "peaks_within": sum(within(pk[n], *EXP_PK[n]) for n in EXP_PK)}
                        r["match"] = best["polynomial"] > 50 and r["peaks_within"] == 4
                        r["partial"] = best["polynomial"] == best.max()
                        (rows if sc == "linear" else rows_v).append(r)
f5b = pd.DataFrame(rows)
print(f5b.to_string(index=False))
print("\nVariant, logarithmic bins (not part of the criterion):")
print(pd.DataFrame(rows_v).to_string(index=False))
M = models(sel("a", "b"), "a")
print("\nMedian resistance by model (P1 a, P4 b, P5 a):", {c: f"{M[c].median():.3g}" for c in M})
print("Distribution distance to the segmentation (two-sample KS statistic), P1 a, P4 b:",
      {c: round(stats.ks_2samp(M[c].dropna(), M.segmentation.dropna()).statistic, 3) for c in M if c != "segmentation"})

**Result: partially reproduced.** The polynomial fit is the closest model for the largest share of
cross sections in every combination: 78.9 to 82.7% against the three fits, and 44.5 to 65.4% when
the annulus competes (annulus 18.5 to 41.8%). It is above 50% in all combinations against the
fits. No combination puts all four peaks within tolerance (at most 2 on linear bins). With P1 (a),
P4 (a) and outliers out, the peaks are 1.31e15 (polynomial, within tolerance), 4.94e14 (ellipse),
1.13e16 (spline, within) and 8.24e14 or 1.10e15 (annulus). The median resistances (P1 a, P4 b)
put the polynomial fit (1.83e15) and the annulus (1.83e15) nearest the segmentation's 4.16e15.

## ellipse-annulus-underestimate

> The resistance distribution of the optimal elliptical annulus is very similar to that of the
> ellipse fit, and both tend to underestimate the resistance of the segmented PVS.

Readings of "underestimate": (a) the median of the per-cross-section ratio to the segmentation is
below 1 for both; (b) the peak and median of both are below the segmentation's. Similarity: the
annulus-over-ellipse ratios of the peaks and of the medians within 10% of 1.87 and 1.61.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for p5 in "ab":
            M = models(d, p5).dropna(subset=["segmentation", "ellipse", "annulus"])
            for p7 in P7:
                for sc in ("linear", "log"):
                    pk = {n: rpeak(M[n], p7, sc) for n in ("segmentation", "ellipse", "annulus")}
                    md_ = {n: M[n].median() for n in pk}
                    ua = (M.ellipse / M.segmentation).median() < 1 and (M.annulus / M.segmentation).median() < 1
                    ub = all(pk[n] < pk["segmentation"] and md_[n] < md_["segmentation"] for n in ("ellipse", "annulus"))
                    rp, rm = pk["annulus"] / pk["ellipse"], md_["annulus"] / md_["ellipse"]
                    sim = within(rp, 1.87, 0.187) and within(rm, 1.61, 0.161)
                    rows.append({"P1": p1, "P4": p4, "P5": p5, "P7": p7, "bins": sc + ("" if sc == "linear" else " (variant)"), "n": len(M),
                                 "med_ratio_ell": round((M.ellipse / M.segmentation).median(), 3),
                                 "med_ratio_ann": round((M.annulus / M.segmentation).median(), 3),
                                 "under_a": ua, "under_b": ub, "ann/ell_peak": round(rp, 3), "ann/ell_median": round(rm, 3),
                                 "similar": sim, "match": (ua or ub) and sim if sc == "linear" else np.nan})
eau = pd.DataFrame(rows)
print(eau.to_string(index=False))
lin = eau[eau.bins == "linear"]
print("\nLinear bins, as in fig5b's criterion: combinations with both underestimating (a or b):", int((lin.under_a | lin.under_b).sum()),
      "of", len(lin), "; with the similarity ratios within 10%:", int(lin.similar.sum()))

**Result: partially reproduced.** Both the ellipse fit and the annulus underestimate the
segmentation in all 16 combinations on linear bins (median per-cross-section ratio 0.264 to 0.301
for the ellipse and 0.461 to 0.634 for the annulus). The annulus-over-ellipse ratios are not
within 10% of the paper's 1.87 and 1.61 in any combination on linear bins (medians 1.767 to
2.731; peaks 0.043 to 2.486). One combination on logarithmic bins, a variant, gives 1.760 and
1.767.

## spline-resistance-five-times

> The spline fit's resistance is considerably larger than the segmentation's, its distribution peak
> about 5 times larger, despite a larger area ratio, because the spline PVS narrows rapidly away
> from the vessel.

Ratio of the spline peak to the segmentation peak, with the bins of the claims it depends on:
reading (a) 5 within 0.5, (b) 6.2 within 0.62; and the median spline resistance above the median
segmented one.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        M = models(d, "a").dropna(subset=["segmentation", "spline"])
        for p7 in P7:
            for sc in ("linear", "log"):
                ratio = rpeak(M.spline, p7, sc) / rpeak(M.segmentation, p7, sc)
                med_ok = M.spline.median() > M.segmentation.median()
                rows.append({"P1": p1, "P4": p4, "P7": p7, "bins": sc, "n": len(M), "peak_ratio": round(ratio, 2),
                             "reading_a": within(ratio, 5, 0.5), "reading_b": within(ratio, 6.2, 0.62),
                             "median_spline": f"{M.spline.median():.3g}", "median_seg": f"{M.segmentation.median():.3g}",
                             "match": (within(ratio, 5, 0.5) or within(ratio, 6.2, 0.62)) and med_ok})
s5 = pd.DataFrame(rows)
print(s5.to_string(index=False))
d = sel("a", "b")
print("\nSpline over segmentation, per cross section (P1 a, P4 b): median", round((d.R_spl_e3 / d.R_seg).median(), 2),
      "; share above 1:", round(100 * ((d.R_spl_e3 / d.R_seg) > 1).mean(), 1), "%")
print("Spline area ratio above the segmentation's (P5 a, fits over pi R1^2):",
      round(100 * ((d.spl_area_e3 / (np.pi * d.R1_e3 ** 2)) > ratio_seg(d, "a")).mean(), 1), "% of cross sections")

**Result: reproduced.** The ratio of the spline peak to the segmentation peak is 5.45 (linear
bins) and 4.94 (logarithmic) with P1 (a), P4 (a) and outliers out, 4.68 with P1 (b), P4 (a), and
5.22 with P1 (b), P4 (b) on logarithmic bins, within 0.5 of 5; in the other 12 combinations it is
0.00 to 4.09. The median spline resistance is above the segmentation's in every combination (1.06e16
against 4.16e15 with P1 a, P4 b), and the spline is the larger in 75.8% of cross sections. The
spline's area ratio exceeds the segmentation's in only 38.8% of cross sections, so the "larger
area ratio" of the statement does not hold here.

## fig7-velocity-profiles

> In a representative cross section, the polynomial fit's velocity profile resembles the
> segmentation's, the ellipse fit gives much higher velocities and the spline fit much lower; these
> trends are generally consistent across cross sections.

Velocity at 500 Pa/m. Readings: (a) maximum velocity, (b) mean velocity. Holds when in more than
half the cross sections the ellipse is faster than the segmentation, the spline slower, and the
polynomial closer to the segmentation than either.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for rd, pre in (("a", "umax"), ("b", "umean")):
            u = pd.DataFrame({m: d[f"{pre}_{s}"] for m, s in (("seg", "seg"), ("ell", "ell_e3"), ("spl", "spl_e3"), ("pol", "pol_e3"))}).dropna()
            fe = 100 * (u.ell > u.seg).mean(); ss = 100 * (u.spl < u.seg).mean()
            dp = (u.pol - u.seg).abs()
            pc = 100 * ((dp < (u.ell - u.seg).abs()) & (dp < (u.spl - u.seg).abs())).mean()
            rows.append({"P1": p1, "P4": p4, "reading": rd, "n": len(u), "ellipse_faster_pct": round(fe, 1),
                         "spline_slower_pct": round(ss, 1), "polynomial_closest_pct": round(pc, 1),
                         "median_seg_um_s": round(1e6 * u.seg.median(), 2), "median_ell": round(1e6 * u.ell.median(), 2),
                         "median_spl": round(1e6 * u.spl.median(), 2), "median_pol": round(1e6 * u.pol.median(), 2),
                         "match": fe > 50 and ss > 50 and pc > 50})
f7 = pd.DataFrame(rows)
print(f7.to_string(index=False))

**Result: reproduced.** In every combination and under both readings the ellipse fit is faster
than the segmentation in 98.7 to 100% of cross sections, the spline slower in 89.5 to 96.7%, and
the polynomial fit closer to the segmentation than either in 66.7 to 75.7%.

## vessel-radius-peak

> The fitted vessel radius distribution peaks at 28 um.

Peak of R1 under P1, P4, P6 and P7; within 2.8 um of 28 um.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            r1 = d[f"R1_e{k}"].dropna()
            for p7 in P7:
                pk = pvs.peak(r1, p7)
                rows.append({"P1": p1, "P4": p4, "P6": k, "P7": p7, "n": len(r1), "peak_um": round(pk, 2),
                             "median_um": round(r1.median(), 2), "match": within(pk, 28, 2.8)})
vr = pd.DataFrame(rows)
print(vr.to_string(index=False))
d = sel("a", "b")
print("\nPer location median R1 (P1 a, P4 b, erosion 3):", d.groupby("location").R1_e3.median().round(1).to_dict())
print("R1 range:", d.R1_e3.min().round(1), "to", d.R1_e3.max().round(1), "um")

**Result: reproduced.** The R1 peak is within 2.8 um of 28 um in 11 of 24 combinations: 28.38
um with P1 (a), P4 (a), erosion 3 and outliers out, and 30.38 with P4 (b), erosion 3 and all
values; the median is 27.7 to 31.4 um in all combinations. Per location the median R1 runs from
17.6 (G) to 49.5 um (E).

## fig8-ellipse-parameters

> The ellipse fit shows an elongated, eccentric PVS with the vessel slightly below its center; R1
> peaks at 28 um and R3 at 35 um, R3/R1 at 0.80, R2/R1 at 2.89, R3/R2 at 0.398, the vertical
> offset Ov at 0.24, and the horizontal offset Oh is centered near zero with a much wider
> distribution than Ov.

R2 and R3 are the semi-axes of the ellipse of the PVS (P9). Offsets in units of (a) R1, (b) R3,
(c) um; Ov positive when the vessel is below the ellipse center. Oh conditions: median within one
Ov peak of zero, IQR of Oh at least twice that of Ov.

In [ ]:
EXP_8 = [("R3", 35, 3.5), ("R3/R1", 0.8, 0.08), ("R2/R1", 2.89, 0.289), ("R3/R2", 0.398, 0.0398), ("Ov", 0.24, 0.024)]
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            r1, r2, r3 = d[f"R1_e{k}"], d[f"R2_e{k}"], d[f"R3_e{k}"]
            for unit in "abc":
                den = r1 if unit == "a" else (r3 if unit == "b" else 1.0)
                ov, oh = (d[f"Ov_e{k}"] / den).dropna(), (d[f"Oh_e{k}"] / den).dropna()
                for p7 in P7:
                    v = {"R3": pvs.peak(r3.dropna(), p7), "R3/R1": pvs.peak((r3 / r1).dropna(), p7), "R2/R1": pvs.peak((r2 / r1).dropna(), p7),
                         "R3/R2": pvs.peak((r3 / r2).dropna(), p7), "Ov": pvs.peak(ov, p7)}
                    oh_c = abs(oh.median()) <= abs(v["Ov"])
                    oh_w = (oh.quantile(0.75) - oh.quantile(0.25)) >= 2 * (ov.quantile(0.75) - ov.quantile(0.25))
                    okv = {n: within(v[n], e, t) for n, e, t in EXP_8}
                    rows.append({"P1": p1, "P4": p4, "P6": k, "units": unit, "P7": p7, **{n: round(x, 3) for n, x in v.items()},
                                 "Oh_median": round(oh.median(), 3), "Oh_IQR": round(oh.quantile(0.75) - oh.quantile(0.25), 3),
                                 "Ov_IQR": round(ov.quantile(0.75) - ov.quantile(0.25), 3), "Oh_centered": oh_c, "Oh_wider": oh_w,
                                 "n_within": sum(okv.values()), "match": all(okv.values()) and oh_c and oh_w,
                                 "ratios_within": okv["R3/R1"] and okv["R2/R1"] and okv["R3/R2"]})
f8 = pd.DataFrame(rows)
print(f8.to_string(index=False))
print("\nCombinations matching:", int(f8.match.sum()), "; with the three ratios within tolerance:", int(f8.ratios_within.sum()), "of", len(f8))
d = sel("a", "b")
print("\nVariant, not used by the criterion: ellipse of PVS and vessel together (P1 a, P4 b, P7 b): R3 peak",
      round(pvs.peak(d.R3u.dropna(), "b"), 1), "um; R3/R1", round(pvs.peak((d.R3u / d.R1_e3).dropna(), "b"), 3),
      "; R2/R1", round(pvs.peak((d.R2u / d.R1_e3).dropna(), "b"), 3), "; R3/R2", round(pvs.peak((d.R3u / d.R2u).dropna(), "b"), 3))

**Result: not reproduced.** No combination matches, and none puts the three ratios within
tolerance. The ellipse of the PVS has R3 peaking at 13.0 to 21.1 um (expected 35), R3/R1 at 0.47
to 0.91, R2/R1 at 1.97 to 4.93 and R3/R2 at 0.219 to 0.254 (expected 0.398). Oh is centred near zero
and its IQR is more than twice that of Ov in most combinations; Ov peaks at 0.30 to 0.90 of R1.
The variant that fits the ellipse to the PVS and vessel together gives R3 29.9 um, R3/R1 0.951,
R2/R1 2.743 and R3/R2 0.431 (P1 a, P4 b); two of the three ratios would be within tolerance. The
question is asked (questions/000491.yaml).

## fig9c-pvs-two-lobes

> The PVS usually consists of two mostly or entirely disconnected lobes on either side of the
> vessel; the polynomial fit's HC (top of PVS above top of vessel) peaks at zero, and when the
> lobes connect the channel is narrow, usually less than 0.5 R1.

Peak of HC/R1 within 0.1 of zero (the claims file's tolerance), and, among cross sections with HC
above zero, more than half with HC below 0.5 R1.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            hc = (d[f"HC_e{k}"] / d[f"R1_e{k}"]).dropna()
            pos = hc[hc > 0]
            for p7 in P7:
                pk = pvs.peak(hc, p7)
                rows.append({"P1": p1, "P4": p4, "P6": k, "P7": p7, "n": len(hc), "peak_HC/R1": round(pk, 3),
                             "median": round(hc.median(), 3), "share_HC>0": round(100 * (hc > 0).mean(), 1),
                             "of_those_below_0.5R1": round(100 * (pos < 0.5).mean(), 1),
                             "match": within(pk, 0.000689, 0.1) and (pos < 0.5).mean() > 0.5})
f9c = pd.DataFrame(rows)
print(f9c.to_string(index=False))

**Result: reproduced.** The peak of HC/R1 is within 0.1 of zero in 15 of 24 combinations (for
example -0.013 with P1 a, P4 b, erosion 3, outliers out; median -0.025), and of the cross sections
with HC above zero, 60.4 to 66.0% have HC below 0.5 R1 in every combination. HC is above zero in
44.4 to 52.8% of cross sections.

## fig9e-pvs-width-asymmetry

> The PVS is about twice as wide as the vessel radius and asymmetric about the vessel; the
> short-side width WS/R1 peaks near 1.1, the long-side width WL/R1 near 1.9, and WS/WL near 0.5.

WS and WL are the shorter and longer of the two side widths of the polynomial fit (vessel edge to
PVS end). Peaks within 0.108, 0.185 and 0.0502 of 1.08, 1.85 and 0.502.

In [ ]:
def widths(d, k):
    w = d[[f"WL_e{k}", f"WR_e{k}"]].values  # left and right side widths
    return pd.DataFrame({"WS": np.nanmin(w, 1), "WL": np.nanmax(w, 1), "R1": d[f"R1_e{k}"].values,
                         "left": w[:, 0], "right": w[:, 1], "location": d.location.values}).dropna()


rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            w = widths(d, k)
            for p7 in P7:
                v = {"WS/R1": pvs.peak(w.WS / w.R1, p7), "WL/R1": pvs.peak(w.WL / w.R1, p7), "WS/WL": pvs.peak(w.WS / w.WL, p7)}
                rows.append({"P1": p1, "P4": p4, "P6": k, "P7": p7, "n": len(w), **{n: round(x, 3) for n, x in v.items()},
                             "median_WS/R1": round((w.WS / w.R1).median(), 3), "median_WL/R1": round((w.WL / w.R1).median(), 3),
                             "median_WS/WL": round((w.WS / w.WL).median(), 3),
                             "match": within(v["WS/R1"], 1.08, 0.108) and within(v["WL/R1"], 1.85, 0.185) and within(v["WS/WL"], 0.502, 0.0502)})
f9e = pd.DataFrame(rows)
print(f9e.to_string(index=False))

**Result: not reproduced.** No combination puts the three peaks within tolerance. WS/R1 peaks at
0.84 to 1.61 (expected 1.08), WL/R1 at 1.82 to 4.18 (expected 1.85) and WS/WL at 0.38 to 0.63
(expected 0.502); the medians are 1.37 to 1.71, 2.67 to 3.31 and 0.585 to 0.616. The two sides
are asymmetric (median WS/WL about 0.6).

## left-right-no-difference

> The PVS widths on the left and right sides of the vessel do not differ significantly.

Readings: (a) Wilcoxon signed-rank test on paired left and right widths over all cross sections;
(b) two-sample Kolmogorov-Smirnov test; (c) Wilcoxon signed-rank test on the per-segment median
widths (14 pairs); alpha 0.05. Left and right are fixed by the direction along the centerline.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            w = widths(d, k)
            pa = stats.wilcoxon(w.left, w.right).pvalue
            pb = stats.ks_2samp(w.left, w.right).pvalue
            m = w.groupby("location")[["left", "right"]].median()
            pc = stats.wilcoxon(m.left, m.right).pvalue
            rows.append({"P1": p1, "P4": p4, "P6": k, "n": len(w), "median_left_minus_right_um": round((w.left - w.right).median(), 2),
                         "p_a": f"{pa:.3g}", "p_b": f"{pb:.3g}", "segments": len(m),
                         "median_seg_diff_um": round((m.left - m.right).median(), 2), "p_c": f"{pc:.3g}",
                         "differences_found": int(pa < 0.05) + int(pb < 0.05) + int(pc < 0.05)})
lr = pd.DataFrame(rows)
print(lr.to_string(index=False))
d = sel("a", "b"); w = widths(d, 3)
print("\nPer location median left and right widths (um), P1 a, P4 b, erosion 3:")
print(w.groupby("location")[["left", "right"]].median().round(1).T.to_string())

**Result: blocked (methods underspecified).** The three readings disagree in every combination:
the paired Wilcoxon test (a) and the Kolmogorov-Smirnov test (b) find a difference (p 2e-16 to
8e-93 and 2e-32 to 1e-121; median left minus right -3.5 to -13.5 um), while the Wilcoxon test on
the per-segment medians (c) does not (p 0.25 to 0.715). The criterion makes this
methods_underspecified. Left and right differ by segment in both directions (left wider in B, C,
D, E, F, L, M; right wider in A, G, H, I, J, K, N).

## fig9a-heights-taper

> The PVS is tallest next to the vessel and narrows far from it; the heights H1/R1, H2/R1 and H3/R1
> peak at 1.34, 0.792 and 0.307, so H1 is about 4 times H3.

Heights of the polynomial fit over R1, (a) both sides pooled, (b) the mean of the two sides.

In [ ]:
def heights(d, k, rd):
    r1 = d[f"R1_e{k}"]
    if rd == "a":
        return pd.DataFrame({h: np.r_[d[f"{h}L_e{k}"] / r1, d[f"{h}R_e{k}"] / r1] for h in ("H1", "H2", "H3")}).dropna()
    return pd.DataFrame({h: (d[f"{h}L_e{k}"] + d[f"{h}R_e{k}"]) / 2 / r1 for h in ("H1", "H2", "H3")}).dropna()


rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            for rd in "ab":
                h = heights(d, k, rd)
                for p7 in P7:
                    pk = {c: pvs.peak(h[c], p7) for c in h}
                    md_ = h.median()
                    rows.append({"P1": p1, "P4": p4, "P6": k, "reading": rd, "P7": p7, "n": len(h),
                                 **{f"pk_{c}": round(pk[c], 3) for c in pk}, **{f"md_{c}": round(md_[c], 3) for c in md_.index},
                                 "match": within(pk["H1"], 1.34, 0.134) and within(pk["H2"], 0.792, 0.0792) and within(pk["H3"], 0.307, 0.0307),
                                 "order_holds": pk["H1"] > pk["H2"] > pk["H3"] and md_.H1 > md_.H2 > md_.H3})
f9a = pd.DataFrame(rows)
print(f9a.to_string(index=False))
print("\nShare of side heights below zero (the line under the polynomial), P1 a, P4 b, erosion 3:",
      (heights(sel("a", "b"), 3, "a") < 0).mean().round(3).to_dict())

**Result: partially reproduced.** No combination puts the three peaks within tolerance. The order
H1 above H2 above H3 holds for the peaks and the medians in 30 of 48 combinations; with outliers out
(P7 b) the H3/R1 peak is 0.246 to 0.427, near the expected 0.307, but H1/R1 is 0.93 to 1.30 and H2/R1
0.80 to 1.32 (expected 1.34 and 0.792). The medians are H1/R1 1.13 to 1.42, H2/R1 1.04 to 1.28 and
H3/R1 0.39 to 0.61, so H1 is 2 to 3 times H3, not 4.

## fig9f-pvs-concave

> The bottom edge of the PVS is most often concave; the distribution of (H2 - H3)/(H1 - H2) peaks
> at 0.108, below the value of 1 that separates concave from convex.

As fig9a, with the same readings: (a) the ratio per side, both sides pooled; (b) the ratio of the
mean heights of the two sides. Peak within 0.0108 of 0.108 and more than half the values below 1.

In [ ]:
rows = []
for p1 in TARGET:
    for p4 in P4:
        d = sel(p1, p4)
        for k in P6:
            for rd in "ab":
                if rd == "a":
                    c = pd.Series(np.r_[(d[f"H2L_e{k}"] - d[f"H3L_e{k}"]) / (d[f"H1L_e{k}"] - d[f"H2L_e{k}"]),
                                        (d[f"H2R_e{k}"] - d[f"H3R_e{k}"]) / (d[f"H1R_e{k}"] - d[f"H2R_e{k}"])])
                else:
                    h = heights(d, k, "b")
                    c = (h.H2 - h.H3) / (h.H1 - h.H2)
                c = c.replace([np.inf, -np.inf], np.nan).dropna()
                for p7 in P7:
                    pk = pvs.peak(c, p7)
                    rows.append({"P1": p1, "P4": p4, "P6": k, "reading": rd, "P7": p7, "n": len(c), "peak": round(pk, 3),
                                 "median": round(c.median(), 3), "share_below_1": round(100 * (c < 1).mean(), 1),
                                 "range": f"{c.min():.3g} to {c.max():.3g}",
                                 "match": within(pk, 0.108, 0.0108) and (c < 1).mean() > 0.5, "partial": (c < 1).mean() > 0.5})
f9f = pd.DataFrame(rows)
print(f9f.to_string(index=False))
print(f"\nNotebook run time {(time.time() - started) / 60:.1f} min")

**Result: partially reproduced.** More than half the values are below 1 in every combination (61.8
to 71.4%; medians 0.072 to 0.491), so the bottom edge is most often concave. The peak is within
0.0108 of 0.108 in no combination; with outliers out it is 0.075 to 1.17.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| cross-section-counts | not reproduced |
| segmentation-validation-overlap | blocked: data missing |
| fig5a-pvs-area-scales-with-vessel-area | not reproduced |
| fig5e-area-ratio-distribution | partially reproduced |
| fig5bc-area-intersegment-variation | reproduced (assumption: P4 b) |
| area-ratio-per-location | not reproduced |
| fig5d-polynomial-closest-area-ratio | partially reproduced |
| table2-fit-area-ratio-peaks | partially reproduced |
| fig6a-resistance-power-law | partially reproduced |
| fig6b-resistance-segmentation-peak | partially reproduced |
| resistance-per-location | not reproduced |
| fig5b-polynomial-closest-resistance | partially reproduced |
| ellipse-annulus-underestimate | partially reproduced |
| spline-resistance-five-times | reproduced |
| fig7-velocity-profiles | reproduced |
| vessel-radius-peak | reproduced |
| fig8-ellipse-parameters | not reproduced |
| fig9c-pvs-two-lobes | reproduced |
| fig9e-pvs-width-asymmetry | not reproduced |
| left-right-no-difference | blocked: methods underspecified |
| fig9a-heights-taper | partially reproduced |
| fig9f-pvs-concave | partially reproduced |

**How the analysis differs from the paper's.** It starts from the deposited segmentations. The
centerline is skimage's 3D thinning, not MATLAB's bwskel with manual removal of points; the
longest path through the skeleton stands in for the manual cleaning. Cross sections are sampled by
linear interpolation of the masks, not with obliqueslice. Pixels in both masks are counted as
vessel. The resistance is solved by finite differences on the pixel grid, not by finite elements.
The resistances of the fits are solved only with the circle fit at the middle erosion (3 pixels),
and reading (b) of P2 is run for the counts only. Every other reading of the claims file is run,
and the tables show each.

**What was found about the dataset.** The earlier published version lacks the PVS masks of L, M
and N. The deposited vessel and PVS masks overlap in 11 of 14 files (up to 344565 voxels at E),
though the Methods say overlaps were removed. At E and F the PVS mask fills 98 to 99% of the
deepest plane. The median area ratio is 7.78 to 9.6 at E, F and G, against 0.72 to 2.22 at the
other segments. L, M and N state a depth step of 0.648 um while their session descriptions say 1 um;
round vessel cross sections are consistent with the stated 0.648 um. The emission wavelengths
put 698 nm on the vessel channel (FITC-dextran), which the Methods give as 525 nm. The 2D time
series used to validate the segmentation are not in the dandiset.